# 🎬 FilmDB — 3-Stage RAG Recommendation Engine
> **A production-grade movie recommendation system built with BGE embeddings, a cross-encoder reranker, and LLaMA-3.3-70B curation.**

| Component | Technology |
|---|---|
| **Corpus** | 95,090 curated movies |
| **Bi-Encoder** | `BAAI/bge-base-en-v1.5` (768-d) |
| **Reranker** | `BAAI/bge-reranker-large` (raw logits) |
| **LLM** | `llama-3.3-70b-versatile` via Groq API |
| **Vector DB** | Qdrant Cloud |
| **Evaluation** | 21 queries across 9 retrieval strategies |

> ⚡ **This notebook runs on CPU only — no GPU or API key needed.**
> All results below are pre-computed from a live T4 GPU run and stored in `showcase_results.json`.
> To interact with the live recommendation engine, visit the [FilmDB web app](https://your-filmdb-url.com).

---

## 1. Pipeline Architecture
The recommendation engine is a **3-stage pipeline** designed to progressively refine candidates:

In [ ]:
import base64
from IPython.display import Image, display, HTML, Markdown

# Note: Mermaid diagrams are rendered via mermaid.ink (requires internet)
def render_mermaid(graph_code):
    b64 = base64.urlsafe_b64encode(graph_code.encode('utf8')).decode('ascii')
    display(HTML(f'<img src="https://mermaid.ink/img/{b64}" style="width:100%; max-width:1400px;">'))

render_mermaid('''
graph TD
    U(["🔍 User Query"]) --> A
    subgraph Stage1 ["Stage 1 — Dense Retrieval"]
        A["BGE-base Bi-Encoder\n768-d Embeddings"] --> B["Cosine Similarity\nTop-50 Candidates"]
    end
    subgraph Stage2 ["Stage 2 — Cross-Encoder Rerank"]
        B --> C["BGE-reranker-large\nRaw Logit Scoring"]
        C --> D["Top-10 Refined"]
    end
    subgraph Stage3 ["Stage 3 — LLM Curation"]
        D --> E["llama-3.3-70b-versatile\nvia Groq API"]
        E --> F(["🎬 Top-5 + Explanation"])
    end
    style Stage1 fill:#1a1a2e,color:#fff
    style Stage2 fill:#16213e,color:#fff
    style Stage3 fill:#0f3460,color:#fff
''')

## 2. Retrieval Strategy Catalog
FilmDB uses **9 distinct retrieval strategies**, each with different filter logic:

In [ ]:
import pandas as pd

strategies = pd.DataFrame([
    {'Strategy': 'S1_Narrative',  'Description': 'Pure semantic plot search',      'Filter': 'None',                    'Example': 'mind-bending movie about dreams'},
    {'Strategy': 'S2_Prestige',   'Description': 'Oscar-winning films only',       'Filter': 'oscar_wins ≥ 1',          'Example': 'sweeping historical epic'},
    {'Strategy': 'S3_Cultural',   'Description': 'Language / regional cinema',     'Filter': 'original_language = X',   'Example': 'Japanese revenge thriller'},
    {'Strategy': 'S4_Platform',   'Description': 'Streaming service filter',       'Filter': 'netflix = 1 (etc.)',      'Example': 'action movie on Netflix'},
    {'Strategy': 'S5_Similarity', 'Description': 'Semantic similarity to a film',  'Filter': 'None',                    'Example': 'movies similar to The Matrix'},
    {'Strategy': 'S6_Director',   'Description': 'Director spotlight / best-of',   'Filter': 'director = X',            'Example': 'best Stanley Kubrick films'},
    {'Strategy': 'S7_Diversity',  'Description': 'Diverse eras & sub-genres',      'Filter': 'Decade bucketing',        'Example': 'diverse list of thrillers'},
    {'Strategy': 'S8_Mood',       'Description': 'Emotional vibe matching',        'Filter': 'None',                    'Example': 'cozy rainy-day movie'},
    {'Strategy': 'S9_HiddenGem',  'Description': 'Obscure high-quality films',     'Filter': 'votes<10k, rating≥7.0',   'Example': 'masterpiece no one talks about'},
])

display(strategies.style
    .set_table_styles([{'selector': 'th', 'props': [('background', '#0f3460'), ('color', 'white'), ('font-size', '13px')]}])
    .hide(axis='index')
)

## 3. Load Precomputed Results
Results from all 21 evaluation queries, pre-run on a Kaggle T4 GPU:

In [ ]:
import json, os
from pathlib import Path

# Dynamically search for the dataset (handles any Kaggle dataset name)
json_paths = list(Path('/kaggle/input').rglob('showcase_results.json'))
json_paths.append(Path('showcase_results.json')) # Local fallback

results = None
for p in json_paths:
    if p.exists():
        with open(p, 'r', encoding='utf-8') as f:
            results = json.load(f)
        print(f'Loaded {len(results)} queries from {p}')
        break

if results is None:
    raise FileNotFoundError('showcase_results.json not found. Please upload it as a Kaggle dataset.')


## 4. Interactive Query Explorer
> **💡 Live interactivity (dropdowns) only works when you are actively running this notebook in a Kaggle session.**
> If you are viewing this as a static output, scroll down to **Section 5** to see the static Plotly charts and **Section 6** for curated highlights.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

STRATEGY_COLORS = {
    'S1_Narrative':  '#e76f51', 'S2_Prestige':  '#2a9d8f', 'S3_Cultural':  '#e9c46a',
    'S4_Platform':   '#264653', 'S5_Similarity':'#118ab2', 'S6_Director':  '#d62828',
    'S7_Diversity':  '#8338ec', 'S8_Mood':      '#06d6a0', 'S9_HiddenGem': '#fb8500',
}

def fmt_table(rows, score_label):
    html = f'<table style="width:100%;border-collapse:collapse;font-size:13px">'
    html += f'<tr style="background:#0f3460;color:white"><th>Title</th><th>Year</th><th>IMDB</th><th>Votes</th><th>{score_label}</th></tr>'
    for i, r in enumerate(rows):
        bg = '#1a1a2e' if i % 2 == 0 else '#16213e'
        html += f'<tr style="background:{bg};color:#eee">'
        html += f'<td>{r["title"]}</td><td>{r["year"]}</td><td>{r["rating"]}</td><td>{r["votes"]:,}</td><td>{r["score"]:.4f}</td>'
        html += '</tr>'
    html += '</table>'
    return html

query_dropdown = widgets.Dropdown(
    options=[(f"{r['id']}: {r['query'][:75]}", i) for i, r in enumerate(results)],
    description='Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%'),
)

output = widgets.Output()

def show_query(change):
    with output:
        clear_output(wait=True)
        r = results[change['new']]
        color = STRATEGY_COLORS.get(r['strategy'], '#888')
        display(HTML(f'<h3 style="color:{color}">🔍 {r["query"]}</h3>'))
        display(HTML(f'<p>Strategy: <code>{r["strategy"]}</code> | Time: <b>{r["elapsed"]}s</b></p>'))
        display(HTML('<h4>Stage 1 — Dense Retrieval (Cosine Similarity)</h4>'))
        display(HTML(fmt_table(r['mode_a'], 'Sim Score')))
        display(HTML('<h4>Stage 2 — Cross-Encoder Rerank</h4>'))
        display(HTML(fmt_table(r['mode_b'], 'Rerank Score')))
        display(HTML('<h4>Stage 3 — LLM Curation (llama-3.3-70b-versatile)</h4>'))
        display(Markdown(r["mode_c"]))

query_dropdown.observe(show_query, names='value')
display(query_dropdown, output)
# Trigger once for initial display
show_query({'new': 0})


## 5. Pipeline Analysis (Plotly)
> These charts are **fully interactive** — hover, zoom, and click even in static view.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Use iframe renderer to ensure charts display in Kaggle's static view
pio.renderers.default = 'iframe'

DARK = '#0f3460'
TEMPLATE = 'plotly_dark'

# Chart 1: Average IMDB Rating — Mode A vs Mode B (reranking quality lift)
rows = []
for r in results:
    a_avg = sum(x['rating'] or 0 for x in r['mode_a']) / len(r['mode_a'])
    b_avg = sum(x['rating'] or 0 for x in r['mode_b']) / len(r['mode_b'])
    rows.append({'Query': r['id'], 'Dense Only': round(a_avg, 2), 'After Rerank': round(b_avg, 2)})

df_lift = pd.DataFrame(rows).melt(id_vars='Query', var_name='Stage', value_name='Avg IMDB Rating')
fig1 = px.bar(df_lift, x='Query', y='Avg IMDB Rating', color='Stage', barmode='group',
    title='📊 Reranker Quality Lift: Avg IMDB Rating Before vs After Reranking',
    color_discrete_map={'Dense Only': '#e76f51', 'After Rerank': '#2a9d8f'},
    template=TEMPLATE, height=420)
fig1.update_layout(legend=dict(orientation='h', y=1.05))
fig1.show()

# Chart 2: Query response time by strategy
df_time = pd.DataFrame([{'Strategy': r['strategy'], 'Time (s)': r['elapsed'], 'Query': r['id']} for r in results])
fig2 = px.strip(df_time, x='Strategy', y='Time (s)', color='Strategy', hover_data=['Query'],
    title='⏱ Query Latency by Strategy (3-stage pipeline)',
    template=TEMPLATE, height=380)
fig2.show()

# Chart 3: Score spread — reranker discriminative power
score_rows = []
for r in results:
    for rec in r['mode_b']:
        score_rows.append({'Query': r['id'], 'Rerank Score': rec['score'], 'Strategy': r['strategy']})

df_scores = pd.DataFrame(score_rows)
fig3 = px.box(df_scores, x='Query', y='Rerank Score', color='Strategy',
    title='🎯 Cross-Encoder Score Distribution per Query (raw logits)',
    template=TEMPLATE, height=420)
fig3.update_layout(showlegend=False)
fig3.show()


## 6. Head-to-Head Highlights
These 4 queries best demonstrate the pipeline's power — **notice how each stage improves the results:**

In [ ]:
HIGHLIGHTS = ['Q02', 'Q15', 'Q06', 'Q08']
TITLES = {
    'Q02': '🌸 Slice of Life — Finding quiet Tokyo family dramas',
    'Q15': '🎬 Director Spotlight — The essential Stanley Kubrick roadmap',
    'Q06': '🎷 Oscar-Winning Jazz Musical — Reranker surfaces Whiplash & La La Land',
    'Q08': '🔫 Indian Gangster Epic — Cross-language retrieval spanning all Indian cinemas',
}

for qid in HIGHLIGHTS:
    r = next(x for x in results if x['id'] == qid)
    display(HTML(f'<hr><h3>{TITLES[qid]}</h3>'))
    display(HTML(f'<p><em>{r["query"]}</em></p>'))
    cols = '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'
    cols += f'<div><b>Stage 1 (Dense)</b>{fmt_table(r["mode_a"][:5], "Sim")}</div>'
    cols += f'<div><b>Stage 2 (Reranked)</b>{fmt_table(r["mode_b"][:5], "Score")}</div>'
    cols += '</div>'
    display(HTML(cols))
    display(HTML(f'<p><b>Stage 3 — LLM Curation:</b></p>'))
    display(Markdown(r["mode_c"]))


## 7. Technical Architecture Deep-Dive

In [ ]:
render_mermaid('''
graph TD
    subgraph Corpus ["📦 Corpus Pipeline (95,090 films)"]
        A["8 Raw Parquet Layers\n347k base movies"] --> B["Data Curation\nDedup, score norm, genre filter"]
        B --> C["Embedding Text Template\nTitle + Year + Genre + Plot"]
        C --> D["BGE-base Embedding\n768-d float32 vectors"]
        D --> E["Qdrant Cloud\nCOSINE distance, 14 payload indices"]
    end
    subgraph Quality ["🛡 Quality Guards"]
        F["Global Guard: year>0, votes>0"]
        G["HiddenGem: 100≤votes≤10k, rating≥7.0"]
        H["Gem Score Blend: 60% gem + 40% semantic"]
    end
    subgraph Eval ["📊 Evaluation (21 queries)"]
        I["6 Retrieval Strategies"] --> J["Mode A: Dense"]
        J --> K["Mode B: +Reranker"]
        K --> L["Mode C: +LLM"]
    end
    E ~~~ F
    H ~~~ I
''')


---
## 🙏 Acknowledgements
- **IMDB Non-Commercial Dataset** — base film metadata
- **Groq API** — `llama-3.3-70b-versatile` inference
- **Qdrant Cloud** — managed vector database
- **BAAI** — `bge-base-en-v1.5` and `bge-reranker-large` models

If you found this notebook useful, please **upvote** ⬆️ and feel free to fork it!

> 💬 Questions? Drop a comment below or open an issue on GitHub.
